In [18]:


import json
import re
from pathlib import Path
from typing import Optional, cast
from tqdm import tqdm
import mlflow
from mlflow.entities import Trace, AssessmentSource, AssessmentSourceType
from loguru import logger

MLFLOW_TRACKING_URI = "http://100.113.186.28:5000"
EXPERIMENT_NAME = "vds-agent-validation"
EXPERIMENT_ID = "6"
TRACE_FILTER = 'trace.text LIKE "%I want%"'

BASE_DIR = Path("/home/tinhanhnguyen/Desktop/HK8/Capstone/CAPSTONE_PROJECT/videodeepsearch")
EVAL_RECORDS_PATH = BASE_DIR / "local/mlflow_eval_records.json"
PREDICTED_SEGMENTS_PATH = BASE_DIR / "test/notebooks/extracted_segments.json"
OUTPUT_DIR = BASE_DIR / "test/notebooks/analysis_results"

Intervals = list[tuple[float, float]]
SegmentMap = dict[str, Intervals]



def parse_timestamp(timestamp: str) -> float:
    """Convert HH:MM:SS, MM:SS, or SS to seconds."""
    parts = timestamp.split(":")
    if len(parts) == 3:
        h, m, s = parts
        return int(h) * 3600 + int(m) * 60 + float(s)
    if len(parts) == 2:
        m, s = parts
        return int(m) * 60 + float(s)
    return float(timestamp)


def seconds_to_timestamp(seconds: float) -> str:
    """Convert seconds to HH:MM:SS."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def merge_intervals(intervals: Intervals) -> Intervals:
    if not intervals:
        return []

    sorted_ivs = sorted(intervals, key=lambda x: x[0])
    merged_ivs = [sorted_ivs[0]]

    for start, end in sorted_ivs[1:]:
        prev_start, prev_end = merged_ivs[-1]
        if start <= prev_end:
            merged_ivs[-1] = (prev_start, max(prev_end, end))
        else:
            merged_ivs.append((start, end))

    return merged_ivs


def interval_duration(intervals: Intervals) -> float:
    return sum(end - start for start, end in intervals)


def intersect_intervals(a: Intervals, b: Intervals) -> Intervals:
    """Intersection of two sorted, merged interval lists."""
    result, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        start = max(a[i][0], b[j][0])
        end = min(a[i][1], b[j][1])
        if start < end:
            result.append((start, end))
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return result


def union_intervals(a: Intervals, b: Intervals) -> Intervals:
    return merge_intervals(a + b)


def temporal_iou(gt: Intervals, pred: Intervals) -> float:
    """Temporal IoU for a single video."""
    if not gt or not pred:
        return 0.0
    gt_m, pred_m = merge_intervals(gt), merge_intervals(pred)
    union_dur = interval_duration(union_intervals(gt_m, pred_m))
    
    print(f"{gt_m=}")
    print(f"{pred_m=}")
    if union_dur == 0:
        return 0.0
    return interval_duration(intersect_intervals(gt_m, pred_m)) / union_dur


def compute_mtgs(gt: SegmentMap, pred: SegmentMap) -> float:
    """Mean temporal IoU over matched (intersection) video IDs."""
    matched = set(gt) & set(pred)
    if not matched:
        return 0.0
    return sum(temporal_iou(gt[vid], pred[vid]) for vid in matched) / len(matched)


def compute_video_recall(gt: SegmentMap, pred: SegmentMap) -> float:
    """
    VideoRecall = percentage of ground truth video IDs that were correctly predicted.

    Formula: |V_M| / |V_G| * 100
    """
    gt_videos = set(gt.keys())
    if len(gt_videos) == 0:
        return 100.0

    matched = gt_videos & set(pred.keys())
    return (len(matched) / len(gt_videos)) * 100.0

In [19]:
pred = {
    "69d1fa20846a50051062bee1": [
      [
        "00:06:13",
        "00:06:19"
      ],
      [
        "00:06:21",
        "00:06:25"
      ],
      [
        "00:06:25",
        "00:06:31"
      ],
      [
        "00:06:39",
        "00:06:50"
      ],
      [
        "00:07:13",
        "00:07:28"
      ],
      [
        "00:05:41",
        "00:07:17"
      ],
      [
        "00:06:45",
        "00:07:39"
      ]
    ]
}

expected = {
    "69d1fa20846a50051062bee1": [
      [
        "00:06:13",
        "00:06:39"
      ],
    ]
}

def turn_to_sec(record: dict) -> SegmentMap:
    for key, val in record.items():
        for index, (start, end) in enumerate(val):
            start_sec = parse_timestamp(start)
            end_sec = parse_timestamp(end)
            record[key][index] = (start_sec, end_sec)
    return record



In [20]:

compute_mtgs(turn_to_sec(expected), turn_to_sec(pred))

gt_m=[(373.0, 399.0)]
pred_m=[(341.0, 459.0)]


0.22033898305084745

In [21]:
pred

{'69d1fa20846a50051062bee1': [(373.0, 379.0),
  (381.0, 385.0),
  (385.0, 391.0),
  (399.0, 410.0),
  (433.0, 448.0),
  (341.0, 437.0),
  (405.0, 459.0)]}